# Does plot improve revenue prediction?

Compare plot-only and metadata-plus-plot models on later releases. The primary target is log recorded revenue. CPI-adjusted revenue is a sensitivity under a common USD assumption.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import NMF
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from corpus import CLEAN_PATH

USE_EMBEDDINGS = os.getenv("MOVIE_SENTENCE_EMBEDDINGS") == "1"
if USE_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer


In [ ]:
FOLDS = [(2005, 2008), (2009, 2011), (2012, 2013), (2014, 2016)]
FINAL_WINDOW = "2014–2016"
TARGETS = ["revenue", "revenue_cpi_2016"]
PRODUCTION_FIELDS = ["production_countries", "production_companies"]
GENRES = ["Drama", "Comedy", "Thriller", "Action", "Romance"]
TOKEN_PATTERN = r"(?u)\b[^\W\d_]{2,}\b"

movies = pd.read_csv(CLEAN_PATH)
labeled = movies.dropna(subset=["revenue", "release_year"]).copy()
labeled["normalized_plot"] = labeled["overview"].str.casefold()

for field in ["genres", *PRODUCTION_FIELDS]:
    labeled[field] = labeled[field].fillna("[]").map(json.loads)

# Both targets use the same films for a paired comparison.
cohort = labeled.dropna(subset=["revenue_cpi_2016"])

## Optional frozen contextual embeddings

Enable with `MOVIE_SENTENCE_EMBEDDINGS=1` and install the optional encoder as described in the README. Encode intact overviews with the checkpoint's tokenizer. For long overviews, split the model's token IDs into windows and average normalized window embeddings with token-count weights.

In [ ]:
embeddings = None

if USE_EMBEDDINGS:
    encoder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
        device="cpu",
    )
    tokenizer = encoder.tokenizer
    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
    window_size = encoder.max_seq_length - special_tokens

    chunks = []
    owners = []
    weights = []

    for row, overview in enumerate(labeled["overview"]):
        tokens = tokenizer.encode(overview, add_special_tokens=False, verbose=False)
        if len(tokens) <= window_size:
            chunks.append(overview)
            owners.append(row)
            weights.append(max(1, len(tokens)))
            continue

        for start in range(0, len(tokens), window_size):
            window = tokens[start:start + window_size]
            chunks.append(tokenizer.decode(window))
            owners.append(row)
            weights.append(len(window))

In [ ]:
if USE_EMBEDDINGS:
    vectors = encoder.encode(
        chunks,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # Normalizing the weighted sum gives the same direction as the weighted mean.
    pooled = np.zeros((len(labeled), vectors.shape[1]))
    weighted_vectors = vectors * np.asarray(weights)[:, None]
    np.add.at(pooled, owners, weighted_vectors)

    norms = np.linalg.norm(pooled, axis=1, keepdims=True)
    pooled /= np.maximum(norms, 1e-12)
    embeddings = pd.DataFrame(pooled, index=labeled.index)

    print(f"Encoded {len(embeddings):,} films with {len(chunks) - len(labeled)} extra windows.")

## Descriptive topics

Fit five topics on the broad text corpus for interpretation. Prediction below fits a separate topic model inside each fold.

In [ ]:
descriptive_vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    stop_words="english",
    min_df=2,
)
corpus_words = descriptive_vectorizer.fit_transform(movies["overview"])
terms = descriptive_vectorizer.get_feature_names_out()

topic_model = NMF(n_components=5, init="nndsvda", random_state=42, max_iter=500)
topic_weights = topic_model.fit_transform(corpus_words)
component_norms = np.linalg.norm(topic_model.components_, axis=1)
dominant_topics = (topic_weights * component_norms).argmax(axis=1)
has_terms = corpus_words.getnnz(axis=1) > 0

topic_rows = []
for topic, component in enumerate(topic_model.components_):
    top_terms = component.argsort()[-10:][::-1]
    assigned_films = (dominant_topics == topic) & has_terms

    topic_rows.append({
        "Topic": topic + 1,
        "Top words": ", ".join(terms[top_terms]),
        "Films": assigned_films.sum(),
    })

print(f"{len(terms):,} terms. {(~has_terms).sum():,} overviews have no retained terms.")
pd.DataFrame(topic_rows).set_index("Topic")

## Model design

Use HW2's alphabetic unigram tokenizer with English stop words and a minimum document frequency of two. Fit five-topic NMF on training-fold TF-IDF. Encode year and runtime numerically, month and genre as indicators, and fit all transforms within the training split. For the metadata extensions use TF-IDF as the common plot representation. Production uses country and company indicators with companies pooled below five training films. Franchise uses a collection flag. Budget uses log budget in the target's units.

In [ ]:
def indicators(
    train: pd.DataFrame,
    test: pd.DataFrame,
    fields: list[str],
) -> tuple[csr_matrix, csr_matrix]:
    """Fit category indicators using training data.

    Args:
        train (pd.DataFrame): Training films.
        test (pd.DataFrame): Evaluation films.
        fields (list[str]): Columns containing category lists.

    Returns:
        tuple[csr_matrix, csr_matrix]: Training and evaluation indicators.
    """
    known_companies = set()
    if "production_companies" in fields:
        frequencies = train["production_companies"].explode().value_counts()
        known_companies = set(frequencies[frequencies.ge(5)].index)

    records = []
    for frame in (train, test):
        frame_records = []
        for row in frame[fields].itertuples(index=False, name=None):
            indicators_for_film = {}
            for field, values in zip(fields, row):
                for value in values:
                    if field == "production_companies" and value not in known_companies:
                        value = "rare"
                    indicators_for_film[f"{field}={value}"] = 1
            frame_records.append(indicators_for_film)
        records.append(frame_records)

    train_records, test_records = records
    vectorizer = DictVectorizer()
    return vectorizer.fit_transform(train_records), vectorizer.transform(test_records)

In [ ]:
def metadata_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
) -> tuple[csr_matrix, csr_matrix]:
    """Encode release year, runtime, month, and genre.

    Args:
        train (pd.DataFrame): Training films.
        test (pd.DataFrame): Evaluation films.

    Returns:
        tuple[csr_matrix, csr_matrix]: Training and evaluation metadata features.
    """
    numeric = ["release_year", "runtime"]
    scaler = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    train_numeric = scaler.fit_transform(train[numeric])
    test_numeric = scaler.transform(test[numeric])

    months = np.eye(12)
    train_months = months[train["release_month"].astype(int) - 1]
    test_months = months[test["release_month"].astype(int) - 1]
    train_genres, test_genres = indicators(train, test, ["genres"])

    train_metadata = hstack([train_numeric, train_months, train_genres], format="csr")
    test_metadata = hstack([test_numeric, test_months, test_genres], format="csr")
    return train_metadata, test_metadata


def combine_features(
    *pairs: tuple[csr_matrix, csr_matrix],
) -> tuple[csr_matrix, csr_matrix]:
    """Stack matching training and evaluation feature blocks.

    Args:
        *pairs (tuple[csr_matrix, csr_matrix]): Train and test blocks in order.

    Returns:
        tuple[csr_matrix, csr_matrix]: Combined train and test matrices.
    """
    train_blocks, test_blocks = zip(*pairs)
    return hstack(train_blocks, format="csr"), hstack(test_blocks, format="csr")

In [ ]:
def plot_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
    *,
    core: bool,
) -> tuple[dict[str, tuple[csr_matrix, csr_matrix]], int]:
    """Encode overviews for the core or metadata extension comparison.

    Args:
        train (pd.DataFrame): Training films.
        test (pd.DataFrame): Evaluation films.
        core (bool): Include NMF and any enabled sentence embeddings.

    Returns:
        tuple[dict, int]: Feature blocks and the fitted vocabulary size.
    """
    vectorizer = TfidfVectorizer(
        token_pattern=TOKEN_PATTERN,
        stop_words="english",
        min_df=2,
    )
    train_words = vectorizer.fit_transform(train["overview"])
    test_words = vectorizer.transform(test["overview"])
    features = {"TF-IDF": (train_words, test_words)}

    if core:
        nmf = NMF(n_components=5, init="nndsvda", random_state=42, max_iter=500)
        train_topics = nmf.fit_transform(train_words)
        test_topics = nmf.transform(test_words)
        features["NMF"] = (csr_matrix(train_topics), csr_matrix(test_topics))

        if embeddings is not None:
            train_vectors = embeddings.loc[train.index].to_numpy()
            test_vectors = embeddings.loc[test.index].to_numpy()
            features["Sentence embedding"] = (
                csr_matrix(train_vectors),
                csr_matrix(test_vectors),
            )

    return features, len(vectorizer.vocabulary_)

In [ ]:
def extension_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
    block: str,
) -> tuple[csr_matrix, csr_matrix]:
    """Encode one additional metadata block for matched-cohort comparisons.

    Args:
        train (pd.DataFrame): Training films eligible for the block.
        test (pd.DataFrame): Evaluation films eligible for the block.
        block (str): Production, Franchise, or Budget.

    Returns:
        tuple[csr_matrix, csr_matrix]: Training and evaluation features.
    """
    if block == "Production":
        return indicators(train, test, PRODUCTION_FIELDS)

    if block == "Franchise":
        train_flag = train[["collection_id"]].notna().astype(float)
        test_flag = test[["collection_id"]].notna().astype(float)
        return csr_matrix(train_flag), csr_matrix(test_flag)

    train_budget = np.log(train[["budget"]])
    test_budget = np.log(test[["budget"]])
    scaler = StandardScaler()
    return (
        csr_matrix(scaler.fit_transform(train_budget)),
        csr_matrix(scaler.transform(test_budget)),
    )

## Fit and score by release window

All models use fixed ridge regularization. Each fold uses earlier films for training and drops exact matches to evaluation overviews. In the optional metadata comparisons, TF-IDF is the common plot representation.

In [ ]:
def compare(
    films: pd.DataFrame,
    target: str,
    block: str = "Core",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Score the chosen comparison across chronological folds.

    Args:
        films (pd.DataFrame): Films eligible for the comparison.
        target (str): Recorded or CPI-adjusted revenue column.
        block (str): Core, Production, Franchise, or Budget.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: Film-level errors and fold counts.
    """
    errors = []
    diagnostics = []

    for start, end in FOLDS:
        test = films.loc[films["release_year"].between(start, end)]
        earlier = films["release_year"].lt(start)
        duplicate_plot = films["normalized_plot"].isin(test["normalized_plot"])
        train = films.loc[earlier & ~duplicate_plot]

        metadata = metadata_features(train, test)
        plots, vocabulary_size = plot_features(train, test, core=block == "Core")
        models = {"Metadata": metadata}

        for name, features in plots.items():
            if block == "Core":
                models[f"Plot: {name}"] = features
            models[f"Metadata + {name}"] = combine_features(metadata, features)

        if block != "Core":
            extra = extension_features(train, test, block)
            extended = combine_features(metadata, extra)
            models[f"Metadata + {block}"] = extended
            models[f"Metadata + {block} + TF-IDF"] = combine_features(
                extended, plots["TF-IDF"]
            )

        y_train = np.log(train[target].to_numpy(dtype=float))
        y_test = np.log(test[target].to_numpy(dtype=float))
        predictions = {"Median": np.full(len(test), np.median(y_train))}

        for name, (train_features, test_features) in models.items():
            model = Ridge(alpha=1.0, solver="lsqr")
            model.fit(train_features, y_train)
            predictions[name] = model.predict(test_features)

        window = f"{start}–{end}"
        for name, prediction in predictions.items():
            residual = prediction - y_test
            errors.append(pd.DataFrame({
                "movie_id": test["movie_id"],
                "Window": window,
                "Model": name,
                "Absolute error": np.abs(residual),
                "Squared error": residual**2,
            }))

        diagnostics.append({
            "Window": window,
            "Training films": len(train),
            "Evaluation films": len(test),
            "Vocabulary": vocabulary_size,
        })

    return pd.concat(errors, ignore_index=True), pd.DataFrame(diagnostics)

## Main comparison

Scores are in log units. The final window uses 728 films.

In [ ]:
core = {}
for target in TARGETS:
    core[target], folds = compare(cohort, target)

folds.set_index("Window")

In [ ]:
holdout_tables = {}
for target, errors in core.items():
    final_window = errors.loc[errors["Window"].eq(FINAL_WINDOW)]
    by_model = final_window.groupby("Model")
    holdout_tables[target] = pd.DataFrame({
        "MAE": by_model["Absolute error"].mean(),
        "RMSE": by_model["Squared error"].mean().pow(0.5),
    })

holdout_scores = pd.concat(holdout_tables, axis=1)
holdout_scores = holdout_scores.rename(
    columns={"revenue": "Recorded revenue",
             "revenue_cpi_2016": "CPI-adjusted sensitivity"},
    level=0,
)
holdout_scores.sort_values(("Recorded revenue", "MAE")).round(3)

In [ ]:
recorded = core["revenue"]
fold_mae = recorded.pivot_table(
    index="Window", columns="Model", values="Absolute error"
)
fold_mae.round(3)

## Where does plot help?

Each entry is the change in log MAE when plot is added to metadata. Negative values favor adding plot. Genre groups overlap. Show groups with at least 30 films.

In [ ]:
paired = recorded.pivot(
    index=["movie_id", "Window"], columns="Model", values="Absolute error"
)
plot_models = paired.columns[paired.columns.str.startswith("Metadata + ")]
deltas = paired[plot_models].sub(paired["Metadata"], axis=0)
deltas.columns = deltas.columns.str.removeprefix("Metadata + ")

deltas.groupby("Window").mean().round(3)

In [ ]:
film_genres = labeled[["movie_id", "genres"]]
by_genre = deltas.reset_index().merge(film_genres, on="movie_id").explode("genres")
by_genre = by_genre.loc[by_genre["genres"].isin(GENRES)]
representations = deltas.columns.tolist()

for grouping in (["genres"], ["Window", "genres"]):
    grouped = by_genre.groupby(grouping)
    summary = grouped[representations].mean()
    summary.insert(0, "Films", grouped.size())
    display(summary.loc[summary["Films"].ge(30)].round(3))

## Additional metadata

Compare production, franchise, and budget separately on their eligible films. Each comparison includes the baseline and TF-IDF controls on those same films.

In [ ]:
has_production = cohort[PRODUCTION_FIELDS].map(bool).all(axis=1)
extensions = {
    "Production": cohort.loc[has_production],
    "Franchise": cohort,
    "Budget": cohort.dropna(subset=["budget"]),
}

extension_scores = {}
for block, eligible in extensions.items():
    scores, _ = compare(eligible, "revenue", block)
    final_window = scores.loc[scores["Window"].eq(FINAL_WINDOW)]
    extension_scores[block] = final_window.groupby("Model")["Absolute error"].agg(
        Films="size",
        MAE="mean",
    )

pd.concat(extension_scores, names=["Comparison"]).round(3)